# 13 — Development-derived threshold analysis

Thresholds are selected using train + validation predictions only, then applied unchanged to the temporal test set.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import yaml
import pandas as pd
from icare_risk.evaluation.thresholds import threshold_analysis

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.parquet'
predictions = pd.read_parquet(prediction_path)
target = config['columns']['target']
prediction_columns = [c for c in predictions.columns if c.endswith('_prob')]
predictions[target] = predictions[target].map({
    **{str(v): 1 for v in config['target_mapping']['positive_values']},
    **{str(v): 0 for v in config['target_mapping']['negative_values']},
}).fillna(pd.to_numeric(predictions[target], errors='coerce'))
predictions = predictions[predictions[target].isin([0, 1])].copy()

selected, test_results = threshold_analysis(
    predictions, target, prediction_columns,
    target_sensitivities=(0.80, 0.90),
)
display(selected)
display(test_results)

,model,prediction_column,threshold_name,threshold
0,lr_kim_score,lr_kim_score_prob,youden,0.501696
1,lr_kim_score,lr_kim_score_prob,sensitivity_0.80,0.501696
2,lr_kim_score,lr_kim_score_prob,sensitivity_0.90,0.488135
3,lr_all_scores,lr_all_scores_prob,youden,0.497299
4,lr_all_scores,lr_all_scores_prob,sensitivity_0.80,0.491947
5,lr_all_scores,lr_all_scores_prob,sensitivity_0.90,0.486596
6,lr_clinical_baseline,lr_clinical_baseline_prob,youden,0.495913
7,lr_clinical_baseline,lr_clinical_baseline_prob,sensitivity_0.80,0.474371
8,lr_clinical_baseline,lr_clinical_baseline_prob,sensitivity_0.90,0.460126
9,xgb_kim_score,xgb_kim_score_prob,youden,0.192403


,model,prediction_column,threshold_name,threshold,n,events,tp,tn,fp,fn,sensitivity,specificity,ppv,npv,predicted_positive_rate
0,lr_kim_score,lr_kim_score_prob,youden,0.501696,2483,508,447,219,1756,61,0.879921,0.110886,0.202905,0.782143,0.887233
1,lr_kim_score,lr_kim_score_prob,sensitivity_0.80,0.501696,2483,508,447,219,1756,61,0.879921,0.110886,0.202905,0.782143,0.887233
2,lr_kim_score,lr_kim_score_prob,sensitivity_0.90,0.488135,2483,508,508,5,1970,0,1.000000,0.002532,0.205004,1.000000,0.997986
3,lr_all_scores,lr_all_scores_prob,youden,0.497299,2483,508,306,765,1210,202,0.602362,0.387342,0.201847,0.791107,0.610552
4,lr_all_scores,lr_all_scores_prob,sensitivity_0.80,0.491947,2483,508,370,516,1459,138,0.728346,0.261266,0.202296,0.788991,0.736609
5,lr_all_scores,lr_all_scores_prob,sensitivity_0.90,0.486596,2483,508,457,203,1772,51,0.899606,0.102785,0.205025,0.799213,0.897704
6,lr_clinical_baseline,lr_clinical_baseline_prob,youden,0.495913,2483,508,286,921,1054,222,0.562992,0.466329,0.213433,0.805774,0.539670
7,lr_clinical_baseline,lr_clinical_baseline_prob,sensitivity_0.80,0.474371,2483,508,388,529,1446,120,0.763780,0.267848,0.211559,0.815100,0.738623
8,lr_clinical_baseline,lr_clinical_baseline_prob,sensitivity_0.90,0.460126,2483,508,431,341,1634,77,0.848425,0.172658,0.208717,0.815789,0.831655
9,xgb_kim_score,xgb_kim_score_prob,youden,0.192403,2483,508,447,219,1756,61,0.879921,0.110886,0.202905,0.782143,0.887233


In [3]:
output_dir = Path(config['outputs']['metrics_directory'])
output_dir.mkdir(parents=True, exist_ok=True)
selected.to_csv(output_dir / 'development_thresholds.csv', index=False)
test_results.to_csv(output_dir / 'fixed_threshold_test_results.csv', index=False)
print(f'Saved results to {output_dir}')

Saved results to results/metrics
